# Análisis exploratorio y limpieza de mediciones de PM2.5 y PM10

El conjunto Evaluaciones Ambientales de Seguimiento — Aire, publicado por OEFA, contiene mediciones ambientales con información espacial y temporal. Este cuaderno examina su estructura, cobertura y calidad y obtiene un subconjunto de concentraciones de PM2.5 y PM10.

El procedimiento comprende identificación de variables, normalización de parámetros y unidades, revisión de valores no cuantificables, validación de concentraciones y ubicación, tratamiento de duplicados y exportación. Se conserva el periodo disponible y la trazabilidad de cada medición.


## 1. Dependencia y acceso a los archivos


In [8]:
%pip -q install duckdb


In [9]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at <DRIVE_MOUNT>; to attempt to forcibly remount, call drive.mount("<DRIVE_MOUNT>", force_remount=True).


In [10]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, shutil, platform, time
import duckdb
import pandas as pd
from IPython.display import display, Markdown

DATA_DIR = Path('/content/drive/MyDrive/Concurrente')
RAW_CSV = DATA_DIR / 'EAS_Aire_0.csv'
if not RAW_CSV.is_file():
    raise FileNotFoundError(f'No existe {RAW_CSV}. Ajusta DATA_DIR a la carpeta del CSV original. No uses el CSV pivotado como sustituto.')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = DATA_DIR / 'Procesado_PM' / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
LOCAL = Path('/content') / ('audit_eas_' + RUN_ID)
LOCAL.mkdir(parents=True, exist_ok=False)
free = shutil.disk_usage('/content').free
if free < 15 * 1024**3:
    raise RuntimeError('Se requieren al menos 15 GiB libres en /content para tablas y temporales.')
con = duckdb.connect(str(LOCAL / 'audit.duckdb'))
con.execute("SET memory_limit='3GB'")
con.execute(f"SET threads={min(os.cpu_count() or 2, 4)}")
con.execute("SET preserve_insertion_order=false")
con.execute("SET temp_directory='" + str(LOCAL / 'spill').replace("'", "''") + "'")
tables = {}
def report(name, sql):
    df = con.execute(sql).df()
    tables[name] = df
    df.to_csv(OUT / (name + '.csv'), index=False)
    print(name)
    display(df.head(50))
    if len(df) > 50:
        print(f'Mostradas 50 de {len(df):,} filas; tabla completa en el CSV.')
    return df
provenance = dict(run_id=RUN_ID, utc=datetime.now(timezone.utc).isoformat(),
    source=str(RAW_CSV), bytes=RAW_CSV.stat().st_size, source_mtime=RAW_CSV.stat().st_mtime,
    python=platform.python_version(), duckdb=duckdb.__version__, cpu=os.cpu_count(),
    scope='Análisis exploratorio y limpieza de PM2.5 y PM10', status='iniciado')
(OUT / 'ejecucion.json').write_text(json.dumps(provenance, indent=2, ensure_ascii=False))
print('CSV original:', RAW_CSV, '\nBytes:', provenance['bytes'], '\nResultados:', OUT)


CSV original: <DATA_DIR>/EAS_Aire_0.csv 
Bytes: 5153626505 
Resultados: <DATA_DIR>/Procesado_PM/20261104T214733_533275Z


## 2. Leer y preservar la identidad de cada medición

Se leen como texto las columnas originales para evitar conversiones silenciosas. Se conserva `Periodo`,
las fechas y horas de inicio y fin, informe y coordenadas. Los errores estructurales del CSV detienen la
lectura: no se utiliza `ignore_errors` ni se descartan líneas sin contarlas.


In [11]:
source = str(RAW_CSV).replace("'", "''")
con.execute(f"""CREATE VIEW raw AS SELECT * FROM read_csv('{source}',
    header=true, auto_detect=true, all_varchar=true, sample_size=100000, ignore_errors=false)""")
schema = con.execute('DESCRIBE raw').df()
display(schema)
schema.to_csv(OUT / '00_esquema_original.csv', index=False)
required = ['Número.de.informe','Nombre.de.la.Evaluación','Nombre.del.punto',
    'Este','Norte','Altitud','Zona','Datum','Periodo','Fecha.inicio','Hora.inicio',
    'Fecha.fin','Hora.fin','Parámetro','Unidad.de.medida','Valor']
missing = sorted(set(required) - set(schema.column_name))
if missing:
    raise ValueError(f'Faltan columnas requeridas: {missing}')
tic = time.perf_counter()
con.execute("""CREATE TABLE mediciones AS
SELECT trim("Número.de.informe") AS informe, trim("Nombre.de.la.Evaluación") AS evaluacion,
    trim("Nombre.del.punto") AS punto, try_cast("Este" AS DOUBLE) AS este,
    try_cast("Norte" AS DOUBLE) AS norte, try_cast("Zona" AS INTEGER) AS zona,
    try_cast("Altitud" AS DOUBLE) AS altitud, trim("Datum") AS datum,
    trim("Periodo") AS periodo, trim("Fecha.inicio") AS fecha_original,
    coalesce(try_cast("Fecha.inicio" AS DATE),
        DATE '1899-12-30' + try_cast(floor(try_cast("Fecha.inicio" AS DOUBLE)) AS INTEGER)) AS fecha,
    trim("Hora.inicio") AS hora_original,
    try_cast(nullif(trim("Hora.inicio"), '-') AS TIME) AS hora,
    trim("Fecha.fin") AS fecha_fin_original, trim("Hora.fin") AS hora_fin_original,
    trim("Parámetro") AS parametro, trim("Unidad.de.medida") AS unidad_original,
    trim("Valor") AS valor_original,
    try_cast(replace(trim("Valor"), ',', '.') AS DOUBLE) AS concentracion
FROM raw""")
print(f'Lectura y conversión: {time.perf_counter()-tic:.1f} segundos')
con.execute("""CREATE VIEW normalizadas AS
SELECT *, CASE
 WHEN regexp_matches(lower(parametro), 'pm *2[.,]5|part[ií]cul.*2[.,]5') THEN 'PM2.5'
 WHEN regexp_matches(lower(parametro), 'pm *10|part[ií]cul.*10') THEN 'PM10'
 ELSE NULL END AS contaminante,
 lower(replace(replace(replace(trim(unidad_original),'μ','µ'),'³','3'),' ','')) AS unidad_norm
FROM mediciones""")
report('01_conteos_originales', """SELECT count(*) AS filas_originales,
    min(fecha) AS primera_fecha, max(fecha) AS ultima_fecha,
    count(*) FILTER(WHERE fecha IS NULL) AS fechas_no_interpretadas,
    count(*) FILTER(WHERE contaminante IS NOT NULL) AS mediciones_PM25_PM10
FROM normalizadas""")
report('02_parametros_y_unidades', """SELECT parametro, contaminante, unidad_original,
    count(*) AS filas FROM normalizadas WHERE contaminante IS NOT NULL GROUP BY ALL ORDER BY filas DESC""")


,column_name,column_type,null,key,default,extra
0,Número.de.informe,VARCHAR,YES,None,None,None
1,Nombre.de.la.Evaluación,VARCHAR,YES,None,None,None
2,Etapa,VARCHAR,YES,None,None,None
3,Componente.ambiental,VARCHAR,YES,None,None,None
4,Procedencia.de.la.Muestra,VARCHAR,YES,None,None,None
5,Procedencia.Especifica.de.la.Muestra,VARCHAR,YES,None,None,None
6,Nombre.del.punto,VARCHAR,YES,None,None,None
7,Este,VARCHAR,YES,None,None,None
8,Norte,VARCHAR,YES,None,None,None
9,Altitud,VARCHAR,YES,None,None,None


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lectura y conversión: 140.4 segundos


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

01_conteos_originales


,filas_originales,primera_fecha,ultima_fecha,fechas_no_interpretadas,mediciones_PM25_PM10
0,11407394,2018-01-01,2024-12-31,0,1580630


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

02_parametros_y_unidades


,parametro,contaminante,unidad_original,filas
0,Material particulado con diámetro menor a 10 m...,PM10,µg/m³,530955
1,Material particulado con diámetro menor a 25 m...,PM2.5,µg/m³,220949
2,"Material particulado con diámetro menor a 2,5 ...",PM2.5,µg/m³,218716
3,Material particulado con diámetro menor a 10 m...,PM10,μg/m³,196246
4,"Material particulado con diámetro menor a 2,5 ...",PM2.5,μg/m³,187121
5,Material particulado con diámetro menor a 10 m...,PM10,μg/m3,157297
6,"Material particulado con diámetro menor a 2,5 ...",PM2.5,μg/m3,50440
7,"Material Particulado PM2,5",PM2.5,μg/m³,7961
8,Material particulado PM10,PM10,μg/m³,7643
9,Material particulado con diámetro menor a 10 m...,PM10,µg/m3,957


,parametro,contaminante,unidad_original,filas
0,Material particulado con diámetro menor a 10 m...,PM10,µg/m³,530955
1,Material particulado con diámetro menor a 25 m...,PM2.5,µg/m³,220949
2,"Material particulado con diámetro menor a 2,5 ...",PM2.5,µg/m³,218716
3,Material particulado con diámetro menor a 10 m...,PM10,μg/m³,196246
4,"Material particulado con diámetro menor a 2,5 ...",PM2.5,μg/m³,187121
5,Material particulado con diámetro menor a 10 m...,PM10,μg/m3,157297
6,"Material particulado con diámetro menor a 2,5 ...",PM2.5,μg/m3,50440
7,"Material Particulado PM2,5",PM2.5,μg/m³,7961
8,Material particulado PM10,PM10,μg/m³,7643
9,Material particulado con diámetro menor a 10 m...,PM10,µg/m3,957


## 3. Calidad y cobertura de PM2.5/PM10

Se conserva todo el periodo disponible y se informa además el subconjunto 2022–2024. Se mantienen
por separado contaminante, unidad y periodo de medición: no se promedian intervalos distintos ni
se fuerza un pivotado para contar volumen. Los valores bajo límite de detección no se convierten
en cero. Los extremos se reportan para revisión y no se eliminan automáticamente.


## Selección de PM2.5 y PM10

Se excluyen gases y meteorología del conjunto de trabajo. Se conserva todo el periodo disponible, sin limitarlo a 2022–2024.


In [12]:
con.execute("""CREATE VIEW candidatas AS SELECT * FROM normalizadas
WHERE contaminante IS NOT NULL""")


## Calidad por contaminante y periodo

Los valores no cuantificables no se sustituyen por cero. Se cuentan faltantes, negativos y unidades diferentes.


In [13]:
report('03_calidad_por_contaminante_periodo', """SELECT contaminante, periodo,
    count(*) AS mediciones,
    count(*) FILTER(WHERE concentracion IS NULL OR NOT isfinite(concentracion)) AS no_numericas_o_no_finitas,
    count(*) FILTER(WHERE concentracion<0) AS negativas,
    count(*) FILTER(WHERE hora IS NULL) AS sin_hora_interpretada,
    count(*) FILTER(WHERE unidad_norm NOT IN ('µg/m3','ug/m3') OR unidad_norm IS NULL) AS otras_unidades
FROM candidatas GROUP BY ALL ORDER BY contaminante, mediciones DESC""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

03_calidad_por_contaminante_periodo


,contaminante,periodo,mediciones,no_numericas_o_no_finitas,negativas,sin_hora_interpretada,otras_unidades
0,PM10,1 hora,857961,238956,0,583083,912
1,PM10,24 horas,36398,10064,0,17097,38
2,PM10,24 Horas,7,0,0,7,0
3,PM2.5,1 hora,658737,185906,0,471459,912
4,PM2.5,24 horas,27520,7832,0,11869,38
5,PM2.5,24 Horas,7,2,0,7,0


,contaminante,periodo,mediciones,no_numericas_o_no_finitas,negativas,sin_hora_interpretada,otras_unidades
0,PM10,1 hora,857961,238956,0,583083,912
1,PM10,24 horas,36398,10064,0,17097,38
2,PM10,24 Horas,7,0,0,7,0
3,PM2.5,1 hora,658737,185906,0,471459,912
4,PM2.5,24 horas,27520,7832,0,11869,38
5,PM2.5,24 Horas,7,2,0,7,0


## Revisión de coordenadas y datum

Las coordenadas se conservan con zona y datum. Este control detecta problemas; no certifica georreferenciación.


In [14]:
report('04_unidades_espaciales', """SELECT zona, datum, count(*) AS filas,
    min(este) AS este_min, max(este) AS este_max,
    min(norte) AS norte_min, max(norte) AS norte_max
FROM candidatas GROUP BY ALL ORDER BY filas DESC""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

04_unidades_espaciales


,zona,datum,filas,este_min,este_max,norte_min,norte_max
0,18,WGS84,590303,271214.0,823956.0,8108673.0,9074886.0
1,17,WGS84,334975,469239.0,769938.0,8990335.0,9494220.0
2,19,WGS84,294409,216594.0,253350.0,8047344.0,8361727.0
3,18,WSG84,150222,289094.0,290677.0,8671337.0,8674301.0
4,17,WGS-84,49150,767988.0,767988.0,8992825.0,8992825.0
5,18,WGS 84,42659,268260.0,361188.0,8671337.0,8946586.0
6,19,WGS 84,42400,189037.0,358735.0,8108673.0,8806535.0
7,17,WGS 84,39614,469239.0,768518.0,8990326.0,9494220.0
8,3600,WGS 84,35450,823956.0,823956.0,8449261.0,8449261.0
9,4285,WGS 84,1448,361831.0,361831.0,8811287.0,8811287.0


,zona,datum,filas,este_min,este_max,norte_min,norte_max
0,18,WGS84,590303,271214.0,823956.0,8108673.0,9074886.0
1,17,WGS84,334975,469239.0,769938.0,8990335.0,9494220.0
2,19,WGS84,294409,216594.0,253350.0,8047344.0,8361727.0
3,18,WSG84,150222,289094.0,290677.0,8671337.0,8674301.0
4,17,WGS-84,49150,767988.0,767988.0,8992825.0,8992825.0
5,18,WGS 84,42659,268260.0,361188.0,8671337.0,8946586.0
6,19,WGS 84,42400,189037.0,358735.0,8108673.0,8806535.0
7,17,WGS 84,39614,469239.0,768518.0,8990326.0,9494220.0
8,3600,WGS 84,35450,823956.0,823956.0,8449261.0,8449261.0
9,4285,WGS 84,1448,361831.0,361831.0,8811287.0,8811287.0


## Identidad de los puntos de monitoreo

Un nombre repetido no identifica por sí solo una estación; se incluyen coordenadas, zona y datum.


In [15]:
report('05_nombres_de_punto_ambiguos', """SELECT punto,
 count(DISTINCT (este,norte,zona,datum)) AS ubicaciones,
 count(*) AS mediciones FROM candidatas GROUP BY punto
 HAVING ubicaciones>1 ORDER BY ubicaciones DESC""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

05_nombres_de_punto_ambiguos


,punto,ubicaciones,mediciones
0,CA-CH-03,2,63850
1,CA-TAL-01,2,56076
2,CA-SB-01,2,54815
3,CA-TA-01,2,43525
4,CA-ESP-01,2,57124
5,CA-HUAR-01,2,56166
6,CA-CG-01,2,3670
7,CA-ROM-01,2,56264
8,CA-AYAS-01,2,56650
9,CA-SOC-01,2,56276


,punto,ubicaciones,mediciones
0,CA-CH-03,2,63850
1,CA-TAL-01,2,56076
2,CA-SB-01,2,54815
3,CA-TA-01,2,43525
4,CA-ESP-01,2,57124
5,CA-HUAR-01,2,56166
6,CA-CG-01,2,3670
7,CA-ROM-01,2,56264
8,CA-AYAS-01,2,56650
9,CA-SOC-01,2,56276


## Reglas de elegibilidad

Se requieren concentración finita no negativa en µg/m³, fecha e identidad espacial. No se descartan extremos automáticamente.


In [16]:
con.execute("""CREATE VIEW elegibles AS SELECT *,
    fecha>=DATE '2022-01-01' AND fecha<DATE '2025-01-01' AS ventana_2022_2024
FROM candidatas WHERE concentracion IS NOT NULL AND isfinite(concentracion)
    AND concentracion>=0 AND unidad_norm IN ('µg/m3','ug/m3')
    AND fecha IS NOT NULL AND punto IS NOT NULL AND punto NOT IN ('','-')
    AND este IS NOT NULL AND norte IS NOT NULL AND isfinite(este) AND isfinite(norte)
    AND este>0 AND norte>0 AND zona IS NOT NULL
    AND datum IS NOT NULL AND datum NOT IN ('','-')""")


## Eliminación de duplicados exactos

La clave incluye informe, estación, periodo y fin de medición. No se elimina una medición por compartir únicamente el lugar o la fecha.


In [17]:
con.execute("""CREATE TABLE limpias AS SELECT DISTINCT informe,evaluacion,punto,
    este,norte,zona,datum,altitud,periodo,fecha,hora,hora_original,
    fecha_fin_original,hora_fin_original,contaminante,concentracion,
    'µg/m³' AS unidad,ventana_2022_2024 FROM elegibles""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Resumen cuantitativo de limpieza

Se registra el número de mediciones después de cada transformación para cuantificar las exclusiones y documentar la composición del conjunto resultante.


In [18]:
report('06_resumen_limpieza', """SELECT 'original' AS etapa,count(*) AS filas FROM mediciones
UNION ALL SELECT 'PM2.5 y PM10 antes de limpiar',count(*) FROM candidatas
UNION ALL SELECT 'elegibles antes de duplicados',count(*) FROM elegibles
UNION ALL SELECT 'limpias con periodo e identidad',count(*) FROM limpias
UNION ALL SELECT 'limpias 2022-2024',count(*) FROM limpias WHERE ventana_2022_2024""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

06_resumen_limpieza


,etapa,filas
0,original,11407394
1,PM2.5 y PM10 antes de limpiar,1580630
2,elegibles antes de duplicados,1135970
3,limpias con periodo e identidad,1135970
4,limpias 2022-2024,927153


,etapa,filas
0,original,11407394
1,PM2.5 y PM10 antes de limpiar,1580630
2,elegibles antes de duplicados,1135970
3,limpias con periodo e identidad,1135970
4,limpias 2022-2024,927153


## Distribución temporal

Se muestran todos los años y periodos para justificar cobertura.


In [19]:
report('07_cobertura_anual', """SELECT year(fecha) AS anio,contaminante,periodo,
    count(*) AS mediciones, count(DISTINCT (punto,este,norte,zona,datum)) AS estaciones
FROM limpias GROUP BY ALL ORDER BY anio,contaminante,periodo""")


07_cobertura_anual


,anio,contaminante,periodo,mediciones,estaciones
0,2018,PM10,1 hora,23007,4
1,2018,PM10,24 horas,1086,7
2,2019,PM10,1 hora,34752,5
3,2019,PM10,24 Horas,7,1
4,2019,PM10,24 horas,1666,9
5,2019,PM2.5,1 hora,6048,1
6,2019,PM2.5,24 Horas,5,1
7,2019,PM2.5,24 horas,337,3
8,2020,PM10,1 hora,49968,6
9,2020,PM10,24 horas,2227,11


,anio,contaminante,periodo,mediciones,estaciones
0,2018,PM10,1 hora,23007,4
1,2018,PM10,24 horas,1086,7
2,2019,PM10,1 hora,34752,5
3,2019,PM10,24 Horas,7,1
4,2019,PM10,24 horas,1666,9
5,2019,PM2.5,1 hora,6048,1
6,2019,PM2.5,24 Horas,5,1
7,2019,PM2.5,24 horas,337,3
8,2020,PM10,1 hora,49968,6
9,2020,PM10,24 horas,2227,11


## Distribución de concentraciones

Se revisan máximos y percentiles antes de decidir sobre extremos. Un extremo estadístico no demuestra error.


In [20]:
report('08_concentraciones', """SELECT contaminante,count(*) AS mediciones,
 min(concentracion) AS minimo,median(concentracion) AS mediana,
 quantile_cont(concentracion,0.99) AS p99,max(concentracion) AS maximo
FROM limpias GROUP BY contaminante ORDER BY contaminante""")


08_concentraciones


,contaminante,mediciones,minimo,mediana,p99,maximo
0,PM10,644396,0.0,20.88,238.1505,6024.55
1,PM2.5,491574,0.0,11.09,134.8800,1256.30


,contaminante,mediciones,minimo,mediana,p99,maximo
0,PM10,644396,0.0,20.88,238.1505,6024.55
1,PM2.5,491574,0.0,11.09,134.8800,1256.30


## Comparación de la composición del conjunto

Se comparan las filas de la fuente completa, las mediciones de PM2.5 y PM10 seleccionadas y las mediciones conservadas después de limpiar. Los conteos se desglosan por contaminante y año.


In [21]:
volumenes = report('09_composicion_del_dataset', """
SELECT 'CSV fuente completo (todos los parámetros)' AS conjunto, count(*) AS registros FROM mediciones
UNION ALL
SELECT 'PM2.5 y PM10 en entrada, antes de limpiar',count(*) FROM candidatas
UNION ALL
SELECT 'PM2.5 y PM10 limpios, todo el periodo',count(*) FROM limpias
UNION ALL
SELECT 'PM2.5 y PM10 limpios, 2022-2024',count(*)
FROM limpias WHERE ventana_2022_2024
""")
report('10_PM_por_contaminante', """
SELECT 'entrada' AS etapa, contaminante, count(*) AS registros
FROM candidatas GROUP BY contaminante
UNION ALL
SELECT 'limpio',contaminante,count(*) FROM limpias GROUP BY contaminante
ORDER BY contaminante, etapa
""")
report('11_PM_por_anio_etapa', """
SELECT 'entrada' AS etapa,year(fecha) AS anio,contaminante,count(*) AS registros
FROM candidatas GROUP BY ALL
UNION ALL
SELECT 'limpio',year(fecha),contaminante,count(*) FROM limpias GROUP BY ALL
ORDER BY anio,contaminante,etapa
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

09_composicion_del_dataset


,conjunto,registros
0,CSV fuente completo (todos los parámetros),11407394
1,"PM2.5 y PM10 en entrada, antes de limpiar",1580630
2,"PM2.5 y PM10 limpios, todo el periodo",1135970
3,"PM2.5 y PM10 limpios, 2022-2024",927153


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

10_PM_por_contaminante


,etapa,contaminante,registros
0,entrada,PM10,894366
1,limpio,PM10,644396
2,entrada,PM2.5,686264
3,limpio,PM2.5,491574


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

11_PM_por_anio_etapa


,etapa,anio,contaminante,registros
0,entrada,2018,PM10,24095
1,limpio,2018,PM10,24093
2,entrada,2019,PM10,36425
3,limpio,2019,PM10,36425
4,entrada,2019,PM2.5,6392
5,limpio,2019,PM2.5,6390
6,entrada,2020,PM10,52195
7,limpio,2020,PM10,52195
8,entrada,2020,PM2.5,17785
9,limpio,2020,PM2.5,17785


,etapa,anio,contaminante,registros
0,entrada,2018,PM10,24095
1,limpio,2018,PM10,24093
2,entrada,2019,PM10,36425
3,limpio,2019,PM10,36425
4,entrada,2019,PM2.5,6392
5,limpio,2019,PM2.5,6390
6,entrada,2020,PM10,52195
7,limpio,2020,PM10,52195
8,entrada,2020,PM2.5,17785
9,limpio,2020,PM2.5,17785


## Síntesis del análisis exploratorio

Se resumen la cobertura temporal, la cantidad de mediciones por contaminante y la proporción de registros conservados. Los resultados describen los datos disponibles; no representan una evaluación predictiva.


## Revisión de valores extremos y exportación

Se conservan las veinte concentraciones más altas por contaminante para su inspección. Se exportan los datos limpios en CSV y Parquet, las tablas de análisis y un manifiesto con la procedencia y el número de registros.


In [22]:
report('12_extremos_por_contaminante', """SELECT contaminante,punto,fecha,hora,periodo,concentracion FROM limpias QUALIFY row_number() OVER(PARTITION BY contaminante ORDER BY concentracion DESC)<=20""")
clean_csv = OUT / 'PM25_PM10_limpio.csv'
clean_parquet = OUT / 'PM25_PM10_limpio.parquet'
def sqlpath(p): return str(p).replace("'", "''")
con.execute("COPY limpias TO '"+sqlpath(clean_csv)+"' (HEADER, DELIMITER ',')")
con.execute("COPY limpias TO '"+sqlpath(clean_parquet)+"' (FORMAT PARQUET, COMPRESSION ZSTD)")
manifest = {'run_id':RUN_ID,'csv':str(clean_csv),'parquet':str(clean_parquet),'filas':int(con.execute('SELECT count(*) FROM limpias').fetchone()[0]),'componentes':['PM10','PM2.5'],'periodo':'todos los años disponibles','unidad':'µg/m³'}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
(DATA_DIR/'Procesado_PM'/'ultima_ejecucion.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
(OUT/'README_datos.md').write_text('Fuente: Evaluaciones Ambientales de Seguimiento - Aire, OEFA. Conservar URL oficial, fecha de descarga y archivo original. Las mediciones conservan periodo, identidad y trazabilidad. Los archivos grandes se mantienen fuera de Git. El cuaderno documenta cómo regenerarlos. \n')
(OUT/'gitignore_sugerido.txt').write_text('data/raw/\ndata/processed/\n*.parquet\n*.duckdb\n*.duckdb.wal\n.ipynb_checkpoints/\n')
for p in [clean_csv,clean_parquet,OUT/'manifest.json']:
    print(p.name,p.stat().st_size,'bytes')


12_extremos_por_contaminante


,contaminante,punto,fecha,hora,periodo,concentracion
0,PM10,CA-PISCO-01,2023-09-12,13:00:00,1 hora,6024.55
1,PM10,CA-ESP-01,2023-08-16,06:00:00,1 hora,3848.78
2,PM10,CA-PISCO-01,2023-09-12,15:00:00,1 hora,3174.59
3,PM10,CA-PISCO-02,2023-09-12,14:00:00,1 hora,3149.94
4,PM10,CA-ESP-01,2023-08-24,05:00:00,1 hora,2916.52
5,PM10,CA-PISCO-01,2023-09-12,14:00:00,1 hora,2881.97
6,PM10,CA-ESP-01,2023-08-19,04:00:00,1 hora,2879.55
7,PM10,CA-CHA-01,2023-07-19,19:00:00,1 hora,2815.68
8,PM10,CA-ESP-01,2024-07-10,<NA>,1 hora,2618.15
9,PM10,CA-ESP-01,2023-06-24,04:00:00,1 hora,2559.41


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PM25_PM10_limpio.csv 240561651 bytes
PM25_PM10_limpio.parquet 4099751 bytes
manifest.json 394 bytes


In [23]:
n_raw = int(con.execute('SELECT count(*) FROM mediciones').fetchone()[0])
n_pm = int(con.execute('SELECT count(*) FROM candidatas').fetchone()[0])
n_clean = int(con.execute('SELECT count(*) FROM limpias').fetchone()[0])
by_pm = dict(con.execute('SELECT contaminante,count(*) FROM limpias GROUP BY contaminante').fetchall())
first_date,last_date = con.execute('SELECT min(fecha),max(fecha) FROM limpias').fetchone()
summary = {'registros_fuente':n_raw,'mediciones_PM_seleccionadas':n_pm,
    'mediciones_PM_limpias':n_clean,'mediciones_por_contaminante':by_pm,
    'fecha_minima':str(first_date),'fecha_maxima':str(last_date),
    'porcentaje_conservado_PM':round(100*n_clean/n_pm,2) if n_pm else None}
(OUT/'resumen_exploratorio.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
text = f"""# Resumen del análisis exploratorio

| Etapa | Registros |
| --- | ---: |
| Fuente completa | {n_raw:,} |
| Mediciones seleccionadas de PM2.5 y PM10 | {n_pm:,} |
| Mediciones limpias | {n_clean:,} |

Se conservaron {by_pm.get('PM10',0):,} mediciones de PM10 y {by_pm.get('PM2.5',0):,} de PM2.5.
El periodo del conjunto limpio comprende desde {first_date} hasta {last_date}.
La proporción conservada respecto de las mediciones PM seleccionadas es {summary['porcentaje_conservado_PM']} %.

Las concentraciones se expresan en µg/m³. Se preservan el periodo de medición, las referencias espaciales y la identidad del informe. La eliminación de duplicados utiliza la clave documentada en el procedimiento. Los valores extremos permanecen disponibles para su revisión.

Las tablas de calidad, cobertura y concentraciones permiten revisar exclusiones y heterogeneidad temporal antes de preparar las observaciones del modelo.
"""
(OUT/'RESUMEN_EXPLORATORIO.md').write_text(text,encoding='utf-8')
provenance.update(status='completado',resultados=summary)
(OUT/'ejecucion.json').write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')
display(Markdown(text))
print('Archivos exportados:',OUT)
con.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Resumen del análisis exploratorio

| Etapa | Registros |
| --- | ---: |
| Fuente completa | 11,407,394 |
| Mediciones seleccionadas de PM2.5 y PM10 | 1,580,630 |
| Mediciones limpias | 1,135,970 |

Se conservaron 644,396 mediciones de PM10 y 491,574 de PM2.5.
El periodo del conjunto limpio comprende desde 2018-01-01 hasta 2024-12-31.
La proporción conservada respecto de las mediciones PM seleccionadas es 71.87 %.

Las concentraciones se expresan en µg/m³. Se preservan el periodo de medición, las referencias espaciales y la identidad del informe. La eliminación de duplicados utiliza la clave documentada en el procedimiento. Los valores extremos permanecen disponibles para su revisión.

Las tablas de calidad, cobertura y concentraciones permiten revisar exclusiones y heterogeneidad temporal antes de preparar las observaciones del modelo.


Archivos exportados: <DATA_DIR>/Procesado_PM/20261104T214733_533275Z
